# Диагностика Llama на исходной задаче balance-scale

6 обращений:3 chat-template и3 текстовых completion с одинаковыми seeds42/43/44.
Исходные625 строк, задача и начальная Python-программа авторов LLM-FE встроены.
Данные H&M и архив Input **не нужны**. T4 + Internet; MODEL_SOURCE hf/unsloth.
Выберите тот же MODEL_SOURCE и MODEL_REVISION, что в проблемном запуске.
Загрузка прежняя: NF4/double quant/FP16, Transformers и temperature0.8.

Это диагностика соблюдения интерфейса, не воспроизведение итоговых метрик статьи.
Начальная функция взята без изменения формул; запрос собран из исходных
domain_head/domain_tail, спецификации,10 train-примеров и версии modify_features_v0.
Выбор примеров/сериализация/способ подачи адаптированы; текстовый completion
не гарантирует совпадения с внутренней обработкой авторского LLM-сервера.
Нет repairs, эволюции или20 успешных наборов. Все ответы и ошибки сохраняются.
Дерево — XGBoost, как в исходной спецификации, не H&M LightGBM.
Используется xgboost-cpu3.0.5 без CUDA; прежняя сборка удаляется перед установкой.
При ошибке контроля или загрузки модели сохраняется ZIP с failure.json и логами.
Оцениваем4-fold CV только на500 train строках первого внешнего fold;125 heldout не оцениваем.
Числовая обработка упрощена; фиксированная версия XGBoost тоже отличается от неизвестного окружения статьи.

Запуск: импортировать .ipynb в отдельный Kaggle Notebook, включить T4/Internet,
выбрать источник/ревизию модели и Run All. Для hf необходим одобренный доступ/HF_TOKEN.
Источники: [код авторов](https://github.com/nikhilsab/LLMFE), [статья](https://arxiv.org/abs/2503.14434).


In [ ]:
import subprocess, sys
# CPU-only wheel: do not mix distributions sharing the xgboost import path.
subprocess.check_call([sys.executable,'-m','pip','uninstall','-y','xgboost','xgboost-cpu'])
# Не переустанавливаем torch/CUDA из готового окружения Kaggle.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.1', 'accelerate==1.10.1', 'bitsandbytes==0.48.1',
    'xgboost-cpu==3.0.5', 'sentencepiece==0.2.1'])


In [ ]:
from pathlib import Path
import os, json, hashlib, time, gc, importlib.metadata
import numpy as np
import pandas as pd
from IPython.display import display, Markdown, Image, FileLink

# Оба варианта скачиваются с Hugging Face; различается издатель репозитория.
MODEL_SOURCE = 'hf'  # 'unsloth' без токена; 'hf' официальный Meta с доступом/HF_TOKEN
MODEL_SOURCES = {
    'hf': {'repo_id':'meta-llama/Llama-3.1-8B-Instruct', 'requires_token':True},
    'unsloth': {'repo_id':'unsloth/Meta-Llama-3.1-8B-Instruct', 'requires_token':False},
}
if MODEL_SOURCE not in MODEL_SOURCES:
    raise ValueError("MODEL_SOURCE должен быть 'hf' или 'unsloth'")
MODEL_ID = MODEL_SOURCES[MODEL_SOURCE]['repo_id']

def resolve_auth(source, environ, load_secret):
    if source not in MODEL_SOURCES:
        raise ValueError("MODEL_SOURCE должен быть 'hf' или 'unsloth'")
    if not MODEL_SOURCES[source]['requires_token']:
        # False запрещает также автоматическое использование старого токена из HF-кэша.
        return False
    value = environ.get('HF_TOKEN')
    if not value:
        try:
            value = load_secret()
        except Exception:
            value = None
    if not value:
        raise RuntimeError('Для hf нужны одобренный доступ Meta и Kaggle Secret HF_TOKEN; либо выберите unsloth')
    return value

# При повторе можно указать SHA из предыдущего run_config.json; пустая строка разрешит текущую ревизию.
MODEL_REVISION = '0e9e39f249a16976918f6564b8830bc894c89659' if MODEL_SOURCE == 'hf' else ''
WORK = Path('/kaggle/working')
INPUT_ROOT = Path('/kaggle/input')
WORK.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(WORK / 'hf_cache')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

CONFIG = {
    'protocol':'balance_scale_llama31_diagnostic_v2_cpu','seed':42,
    'evaluator_distribution':'xgboost-cpu==3.0.5',
    'model_id':MODEL_ID,'model_source':MODEL_SOURCE,
    'quantization':'bnb NF4 4bit/double_quant, FP16 compute',
    'temperature':0.8,'top_p':1.0,'top_k':0,
    'max_input_tokens':6144,'max_new_tokens':2048,
    'formats':['chat','completion'],'seeds':[42,43,44],'max_calls':6,
    'paper_reproduction':False,'heldout_evaluated':False,
}


In [ ]:
MODULES = {'host.py': '"""Small frozen-prompt diagnostic, not an evolutionary reproduction."""\nimport ast\nimport json\nimport os\nfrom pathlib import Path\nimport subprocess\nimport sys\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedKFold\nfrom sklearn.preprocessing import LabelEncoder\n\n\ndef write_json(path,value):\n    Path(path).write_text(json.dumps(value,ensure_ascii=False,indent=2)+\'\\n\',encoding=\'utf-8\')\n\n\ndef pack_results(out,error=None,stage=None):\n    import shutil\n    import traceback\n    out=Path(out)\n    if error is not None:\n        write_json(out/\'failure.json\',{\'status\':\'failed\',\'stage\':stage,\'error\':str(error),\n            \'traceback\':\'\'.join(traceback.format_exception(type(error),error,error.__traceback__)),\n            \'llm_calls_started\':len(list(out.glob(\'*_seed*/request.json\')))})\n    return shutil.make_archive(str(out.parent/(out.name+\'_results\')),\'zip\',root_dir=out)\n\n\ndef initial_source(root):\n    spec=(Path(root)/\'specs/specification_balance-scale.txt\').read_text()\n    return spec.split(\'@equation.evolve\\n\',1)[1]\n\n\ndef build_prompt(root):\n    root=Path(root);spec=(root/\'specs/specification_balance-scale.txt\').read_text()\n    template=ast.literal_eval(ast.parse(spec).body[0].value)\n    frame=pd.read_csv(root/\'data/balance-scale.csv\')\n    y=LabelEncoder().fit_transform(frame.iloc[:,-1])\n    rows,_=next(StratifiedKFold(n_splits=5,shuffle=True,random_state=42).split(frame.iloc[:,:-1],y))\n    train=frame.iloc[rows].reset_index(drop=True)\n    descriptions=\'\\n\'.join(\'- \'+column+\': numerical variable within range [1, 5]\' for column in train.columns[:-1])\n    examples=[]\n    for i in train.sample(10,random_state=42).index:\n        parts=[column+\' is \'+str(train.loc[i,column]) for column in train.columns[:-1]]\n        examples.append(\'If \'+\', \'.join(parts)+\', Then Result is \'+str(y[rows[i]])+\'.\')\n    for key,value in {\'PREFIX\':(root/\'prompts/domain_head.txt\').read_text(),\n                      \'SUFFIX\':(root/\'prompts/domain_tail.txt\').read_text(),\n                      \'FEATURES\':descriptions,\'EXAMPLES\':\'\\n\'.join(examples)}.items():\n        template=template.replace(\'[\'+key+\']\',value)\n    seed=initial_source(root).replace(\'def modify_features(\', \'def modify_features_v0(\')\n    header=\'def modify_features_v1(df_input) -> pd.DataFrame:\\n    """Improved version of `modify_features_v0`. Think and suggest new features."""\\n\'\n    instruction=(\'You are a helpful assistant tasked with discovering new features/ dropping less important feaures \'\n                 \'for the given prediction task. Complete the modify_features function below, \'\n                 \'considering the physical meaning and relationships of inputs.\\n\\n\')\n    return instruction+template+\'\\n\'+seed+\'\\n\'+header\n\n\ndef evaluate(root,directory):\n    directory=Path(directory)\n    if (directory/\'result.json\').exists():return json.loads((directory/\'result.json\').read_text())\n    environment={k:v for k,v in os.environ.items() if not any(t in k.upper() for t in [\'TOKEN\',\'SECRET\',\'API_KEY\'])}\n    environment.update(CUDA_VISIBLE_DEVICES=\'\',OMP_NUM_THREADS=\'1\',OPENBLAS_NUM_THREADS=\'1\',MKL_NUM_THREADS=\'1\')\n    command=[sys.executable,str(Path(__file__).with_name(\'worker.py\')),\'--data\',str(Path(root)/\'data/balance-scale.csv\'),\'--directory\',str(directory)]\n    try:\n        with (directory/\'worker.log\').open(\'w\') as log:\n            completed=subprocess.run(command,env=environment,stdout=log,stderr=subprocess.STDOUT,timeout=90)\n        if completed.returncode or not (directory/\'result.json\').exists():\n            raise RuntimeError((directory/\'worker.log\').read_text()[-3000:])\n        return json.loads((directory/\'result.json\').read_text())\n    except Exception as error:\n        result={\'status\':\'rejected\',\'phase\':\'worker\',\'error\':str(error)}\n        write_json(directory/\'result.json\',result)\n        return result\n\n\ndef controls(root,out):\n    results={}\n    for name,source in {\'raw\':\'def modify_features(df_input):\\n    return df_input.copy()\\n\',\n                        \'authors_seed\':initial_source(root)}.items():\n        directory=Path(out)/name;directory.mkdir(exist_ok=True)\n        (directory/\'response.txt\').write_text(source)\n        result=evaluate(root,directory)\n        if result[\'status\']!=\'valid\':raise RuntimeError(result)\n        results[name]=result\n        print(name,\'train CV accuracy:\',result[\'cv_accuracy\'],flush=True)\n    write_json(Path(out)/\'controls.json\',results)\n    return results\n\n\ndef run_diagnostic(root,out,config,generate):\n    out=Path(out);prompt=build_prompt(root)\n    (out/\'prompt_base.txt\').write_text(prompt)\n    trials=[]\n    for mode in config[\'formats\']:\n        for seed in config[\'seeds\']:\n            directory=out/f\'{mode}_seed{seed}\';directory.mkdir(exist_ok=True)\n            (directory/\'prompt.txt\').write_text(prompt)\n            if (directory/\'trial.json\').exists():\n                trials.append(json.loads((directory/\'trial.json\').read_text()));continue\n            trial={\'format\':mode,\'seed\':seed,\'directory\':str(directory)}\n            try:\n                response_path=directory/\'response.json\'\n                if response_path.exists():response=json.loads(response_path.read_text())\n                else:\n                    if (directory/\'request.json\').exists():raise RuntimeError(\'Interrupted request consumes diagnostic slot; not silently regenerated\')\n                    write_json(directory/\'request.json\',trial)\n                    print(\'LLM call\',len(trials)+1,\'/6:\',mode,\'seed\',seed,flush=True)\n                    response=generate(prompt,seed,mode)\n                    write_json(response_path,response)\n                trial.update({k:v for k,v in response.items() if k not in {\'text\',\'rendered_prompt\'}})\n                (directory/\'response.txt\').write_text(response[\'text\'])\n                (directory/\'rendered_prompt.txt\').write_text(response.get(\'rendered_prompt\',prompt))\n                if response.get(\'truncated\'):raise ValueError(\'Truncated at output token limit\')\n                trial.update(evaluate(root,directory))\n            except Exception as error:trial.update(status=\'rejected\',error=str(error))\n            write_json(directory/\'trial.json\',trial)\n            trials.append(trial);write_json(out/\'trials.json\',trials)\n            print(mode,seed,trial[\'status\'],\'CV:\',trial.get(\'cv_accuracy\'),\'error:\',trial.get(\'error\',\'\'),flush=True)\n    return trials\n\n\ndef report(out,config,trials,control_results):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    out=Path(out);summary=[]\n    for mode in config[\'formats\']:\n        subset=[t for t in trials if t[\'format\']==mode]\n        summary.append({\'format\':mode,\'calls\':len(subset),\'valid\':sum(t[\'status\']==\'valid\' for t in subset),\n                        \'changed_features\':sum(t.get(\'has_feature_change\',False) for t in subset),\n                        \'truncated\':sum(t.get(\'truncated\',False) for t in subset)})\n    fig,axes=plt.subplots(1,2,figsize=(10,4))\n    axes[0].bar([s[\'format\'] for s in summary],[s[\'valid\'] for s in summary]);axes[0].set(ylabel=\'Executable /3 calls\',ylim=(0,3))\n    for mode in config[\'formats\']:\n        values=[t[\'cv_accuracy\'] for t in trials if t[\'format\']==mode and t[\'status\']==\'valid\']\n        if values:axes[1].scatter([mode]*len(values),values,label=mode)\n    for name,color in [(\'raw\',\'gray\'),(\'authors_seed\',\'green\')]:\n        axes[1].axhline(control_results[name][\'cv_accuracy\'],label=name,color=color)\n    axes[1].set_ylabel(\'Train-only4-fold CV accuracy\');axes[1].legend()\n    fig.suptitle(\'Balance-scale diagnostic — NOT paper reproduction / NOT H&M\');fig.tight_layout()\n    fig.savefig(out/\'diagnostic.png\',dpi=150);plt.close(fig)\n    write_json(out/\'summary.json\',{\'diagnostic_only\':True,\'paper_reproduction\':False,\'formats\':summary,\n        \'heldout_evaluated\':False,\'h_and_m_evaluated\':False,\'controls\':control_results})\n    lines=[\'# Balance-scale Llama diagnostic\',\'\',pd.DataFrame(summary).to_string(index=False),\'\',\n           \'No H&M or heldout result. Executable output alone does not establish useful feature discovery.\',\n           \'Chat/completion differences suggest format sensitivity;3 draws do not establish causality.\',\n           \'If both fail, inspect responses and model configuration before more H&M search.\']\n    (out/\'report.md\').write_text(\'\\n\'.join(lines))\n    pd.DataFrame([{k:t.get(k) for k in [\'format\',\'seed\',\'status\',\'cv_accuracy\',\'has_feature_change\',\'error\']} for t in trials]).to_csv(out/\'trials.csv\',index=False)\n    return summary\n', 'worker.py': '"""CPU-only balance-scale feature compliance and four-fold CV diagnostic."""\nimport os\nif __name__==\'__main__\':\n    os.environ[\'CUDA_VISIBLE_DEVICES\']=\'\'\n    for key in [\'OMP_NUM_THREADS\',\'OPENBLAS_NUM_THREADS\',\'MKL_NUM_THREADS\']:\n        os.environ[key]=\'1\'\nimport ast\nimport argparse\nimport json\nfrom pathlib import Path\nimport re\nimport resource\nimport signal\nimport traceback\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import StratifiedKFold\nfrom sklearn.preprocessing import LabelEncoder\nfrom xgboost import XGBClassifier\nimport xgboost\n\nFORBIDDEN={\'open\',\'eval\',\'exec\',\'compile\',\'getattr\',\'setattr\',\'globals\',\'locals\',\'vars\',\'__import__\',\n           \'input\',\'breakpoint\',\'help\',\'dir\',\'type\',\'super\'}\n\n\ndef extract(text):\n    blocks=re.findall(r\'```(?:python)?\\s*\\n(.*?)```\',text,re.S|re.I)\n    source=next((b for b in blocks if \'def modify_features\' in b),text).strip()\n    if not re.search(r\'def modify_features(?:_v\\d+)?\\s*\\(\',source):\n        # Authors sample a continuation after a prefilled function header.\n        source=\'def modify_features(df_input):\\n\'+ \'\\n\'.join(\'    \'+line for line in source.splitlines())\n    tree=ast.parse(source)\n    functions=[n for n in tree.body if isinstance(n,ast.FunctionDef) and re.fullmatch(r\'modify_features(?:_v\\d+)?\',n.name)]\n    if not functions:raise ValueError(\'No modify_features function\')\n    function=functions[-1]\n    if len(function.args.args)!=1 or function.decorator_list:\n        raise ValueError(\'One dataframe argument, no decorators\')\n    function.name=\'modify_features\'\n    for node in ast.walk(function):\n        if isinstance(node,(ast.Import,ast.ImportFrom)):\n            permitted=isinstance(node,ast.Import) and all(a.name in {\'pandas\',\'numpy\'} and a.asname=={\'pandas\':\'pd\',\'numpy\':\'np\'}[a.name] for a in node.names)\n            if not permitted:raise ValueError(\'Only imports pandas as pd / numpy as np are supported\')\n        if isinstance(node,(ast.Global,ast.Nonlocal,ast.ClassDef,ast.With)):\n            raise ValueError(\'No global state, classes or context managers\')\n        if isinstance(node,ast.Name) and (node.id.startswith(\'_\') or node.id in FORBIDDEN):raise ValueError(\'Forbidden name \'+node.id)\n        if isinstance(node,ast.Attribute) and (node.attr.startswith(\'_\') or node.attr.startswith((\'read_\',\'to_\')) and node.attr not in {\'to_numpy\',\'to_frame\',\'to_list\',\'to_dict\'} or node.attr in {\'eval\',\'query\',\'ctypes\',\'save\',\'load\',\'loadtxt\',\'savetxt\',\'fromfile\',\'memmap\'}):\n            raise ValueError(\'Forbidden attribute \'+node.attr)\n    # Only the selected function is compiled. Permit pd/np imports by removing them from its body.\n    class RemoveImports(ast.NodeTransformer):\n        def visit_Import(self,node):return None\n    function=RemoveImports().visit(function)\n    tree=ast.fix_missing_locations(ast.Module(body=[function],type_ignores=[]))\n    return ast.unparse(tree)+\'\\n\'\n\n\ndef transform(source,X):\n    import builtins\n    names=[\'abs\',\'min\',\'max\',\'sum\',\'len\',\'range\',\'enumerate\',\'zip\',\'list\',\'tuple\',\'dict\',\'set\',\n           \'float\',\'int\',\'str\',\'bool\',\'round\',\'isinstance\',\'any\',\'all\',\'ValueError\',\'Exception\']\n    scope={\'pd\':pd,\'np\':np,\'__builtins__\':{n:getattr(builtins,n) for n in names}}\n    exec(compile(source,\'<balance-features>\',\'exec\'),scope)\n    before=X.copy(deep=True)\n    output=scope[\'modify_features\'](X.copy(deep=True))\n    if not isinstance(output,pd.DataFrame) or not output.index.equals(before.index):\n        raise ValueError(\'Return dataframe preserving rows and index\')\n    if not 1<=len(output.columns)<=64 or output.columns.duplicated().any():\n        raise ValueError(\'Require1-64 unique columns\')\n    if any(not pd.api.types.is_numeric_dtype(output[c]) for c in output):raise ValueError(\'Numeric features required\')\n    output=output.astype(float).replace([np.inf,-np.inf],np.nan)\n    if output.isna().all(axis=None):raise ValueError(\'All features are missing\')\n    return output\n\n\ndef run(data,directory):\n    directory=Path(directory);phase=\'parse\'\n    try:\n        build=xgboost.build_info()\n        if build.get(\'USE_CUDA\') is not False:\n            raise RuntimeError(\'Expected xgboost-cpu with USE_CUDA=false; got \'+str(build))\n        source=extract((directory/\'response.txt\').read_text())\n        (directory/\'program.py\').write_text(source)\n        frame=pd.read_csv(data)\n        X=frame.iloc[:,:-1];y=LabelEncoder().fit_transform(frame.iloc[:,-1])\n        # Exactly the first outer fold, as in authors\' five-fold setup; heldout is never evaluated.\n        train,heldout=next(StratifiedKFold(n_splits=5,shuffle=True,random_state=42).split(X,y))\n        X=X.iloc[train].reset_index(drop=True);y=y[train]\n        phase=\'feature_runtime\'\n        output=transform(source,X)\n        extra=[c for c in output if c not in X or not np.array_equal(output[c],X[c],equal_nan=True)]\n        removed=[c for c in X if c not in output]\n        phase=\'cv_training\'\n        scores=[]\n        for a,b in StratifiedKFold(n_splits=4,shuffle=True,random_state=42).split(output,y):\n            model=XGBClassifier(random_state=42,n_jobs=4,tree_method=\'hist\',device=\'cpu\')\n            model.fit(output.iloc[a],y[a])\n            scores.append(float((model.predict(output.iloc[b])==y[b]).mean()))\n        result={\'status\':\'valid\',\'cv_accuracy\':float(np.mean(scores)),\'cv_folds\':scores,\n                \'columns\':list(output),\'changed_or_added_columns\':extra,\'removed_columns\':removed,\n                \'has_feature_change\':bool(extra or removed),\n                \'train_rows\':len(X),\'heldout_rows_not_evaluated\':len(heldout),\'paper_reproduction\':False,\n                \'xgboost_version\':xgboost.__version__,\'xgboost_build_info\':build}\n    except Exception as error:\n        result={\'status\':\'rejected\',\'phase\':phase,\'error\':str(error),\'traceback\':traceback.format_exc()}\n    (directory/\'result.json\').write_text(json.dumps(result,indent=2))\n    print(json.dumps(result),flush=True)\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser();parser.add_argument(\'--data\');parser.add_argument(\'--directory\')\n    args=parser.parse_args()\n    resource.setrlimit(resource.RLIMIT_AS,(4*1024**3,4*1024**3))\n    signal.signal(signal.SIGALRM,lambda *_:(_ for _ in ()).throw(TimeoutError(\'Diagnostic worker exceeded60s\')))\n    signal.alarm(60)\n    run(args.data,args.directory)\n'}
INPUT_FILES = {'LICENSE': 'MIT License\n\nCopyright (c) 2025 Nikhil Abhyankar\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'preprocessing.py': '# Implementation adapted from CAAFE (Hollmann et al., 2023)\n# Original code: https://github.com/noahho/CAAFE/blob/main/caafe/preprocessing.py\n\nimport pandas as pd\nimport copy\nimport numpy as np\nfrom typing import Dict, Optional, Tuple\n\n\ndef create_mappings(df_train: pd.DataFrame) -> Dict[str, Dict[int, str]]:\n    """\n    Creates a dictionary of mappings for categorical columns in the given dataframe.\n\n    Parameters:\n    df_train (pandas.DataFrame): The dataframe to create mappings for.\n\n    Returns:\n    Dict[str, Dict[int, str]]: A dictionary of mappings for categorical columns in the dataframe.\n    """\n    mappings = {}\n    for col in df_train.columns:\n        if (\n            df_train[col].dtype.name == "category"\n            or df_train[col].dtype.name == "object"\n        ):\n            mappings[col] = {v: i for i, v in \n                enumerate(df_train[col].astype("category").cat.categories)\n            }\n    return mappings\n\n\ndef convert_categorical_to_integer_f(column: pd.Series, mapping: Optional[Dict[int, str]] = None) -> pd.Series:\n    """\n    Converts a categorical column to integer values using the given mapping.\n\n    Parameters:\n    column (pandas.Series): The column to convert.\n    mapping (Dict[int, str], optional): The mapping to use for the conversion. Defaults to None.\n\n    Returns:\n    pandas.Series: The converted column.\n    """\n    if mapping is not None:\n        # if column is categorical\n        if column.dtype.name == "category":\n            column = column.cat.add_categories([-1])\n        return column.map(mapping).fillna(-1).astype(int)\n    return column\n\ndef preprocess_dataset(df: pd.DataFrame, mappings: Dict[str, Dict[int, str]]) -> pd.DataFrame:\n    """\n    Converts the categorical columns in the given dataframe to integer values using the given mappings.\n\n    Parameters:\n    df (pandas.DataFrame): The dataframe to convert.\n    mappings (Dict[str, Dict[int, str]]): The mappings to use for the conversion.\n\n    Returns:\n    pandas.DataFrame: The converted dataframe.\n    """\n    df = df.replace([np.inf, -np.inf], np.nan)\n    df = df.apply(\n        lambda col: convert_categorical_to_integer_f(\n            col, mapping=mappings.get(col.name)\n        ),\n        axis=0,\n    )\n    df = df.astype(float)\n\n    return df\n\n\ndef preprocess_datasets(df_train: pd.DataFrame, df_test: Optional[pd.DataFrame], target_column: str, return_mappings: Optional[bool] = False) -> Tuple[pd.DataFrame, Optional[pd.DataFrame], Optional[Dict[str, Dict[int, str]]]]:\n    """\n    Converts the categorical columns in the given training and test dataframes to integer values using mappings created from the training dataframe.\n\n    Parameters:\n    df_train (pandas.DataFrame): The training dataframe to convert.\n    df_test (pandas.DataFrame, optional): The test dataframe to convert. Defaults to None.\n    target_column (str): The name of the target column.\n    return_mappings (bool, optional): Whether to return the mappings used for the conversion. Defaults to False.\n\n    Returns:\n    Tuple[pandas.DataFrame, Optional[pandas.DataFrame], Optional[Dict[str, Dict[int, str]]]]: The converted training dataframe, the converted test dataframe (if it exists), and the mappings used for the conversion (if `return_mappings` is True).\n    """\n    df_train = copy.deepcopy(df_train)\n    df_train = df_train.infer_objects()\n    if df_test is not None:\n        df_test = copy.deepcopy(df_test)\n        df_test = df_test.infer_objects()\n\n    # Create the mappings using the train and test datasets\n    mappings = create_mappings(df_train)\n\n    # Apply the mappings to the train and test datasets\n    non_target = [c for c in df_train.columns if c != target_column]\n    df_train[non_target] = preprocess_dataset(df_train[non_target], mappings)\n\n    if df_test is not None:\n        df_test[non_target] = preprocess_dataset(df_test[non_target], mappings)\n\n    if return_mappings:\n        return df_train, df_test, mappings\n\n    return df_train, df_test\n', 'utils.py': 'import random\nimport numpy as np\nimport pandas as pd\nfrom pandas.api.types import is_string_dtype\n\ndef is_categorical(x):\n    assert type(x) == pd.Series\n    x = x.convert_dtypes()\n    if is_string_dtype(x):\n        return True\n  \n    elif set(x) == {0, 1}:\n        return True\n  \n    elif x.dtype in [int, float, \'Int64\', \'Float64\']:\n        return False\n \n    else:\n        return True\n\n\ndef set_seed(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n\n\ndef serialize(row):\n    target_str = f""\n    for attr_idx, attr_name in enumerate(list(row.index)):\n        if attr_idx == 0:\n            target_str += "If "\n        if attr_idx < len(list(row.index)) - 1:\n            target_str += " is ".join([attr_name, str(row[attr_name]).strip(" .\'").strip(\'"\').strip()])\n            target_str += ", "\n        else:\n            if len(attr_name.strip()) < 2:\n                continue\n            target_str += " Then "\n            target_str += " is ".join([attr_name, str(row[attr_name]).strip(" .\'").strip(\'"\').strip()])\n            target_str += "."\n    return target_str', 'data/balance-scale.csv': 'Left-Weight,Left-Distance,Right-Weight,Right-Distance,Class Name\n1,1,1,1,balanced\n1,1,1,2,right\n1,1,1,3,right\n1,1,1,4,right\n1,1,1,5,right\n1,1,2,1,right\n1,1,2,2,right\n1,1,2,3,right\n1,1,2,4,right\n1,1,2,5,right\n1,1,3,1,right\n1,1,3,2,right\n1,1,3,3,right\n1,1,3,4,right\n1,1,3,5,right\n1,1,4,1,right\n1,1,4,2,right\n1,1,4,3,right\n1,1,4,4,right\n1,1,4,5,right\n1,1,5,1,right\n1,1,5,2,right\n1,1,5,3,right\n1,1,5,4,right\n1,1,5,5,right\n1,2,1,1,left\n1,2,1,2,balanced\n1,2,1,3,right\n1,2,1,4,right\n1,2,1,5,right\n1,2,2,1,balanced\n1,2,2,2,right\n1,2,2,3,right\n1,2,2,4,right\n1,2,2,5,right\n1,2,3,1,right\n1,2,3,2,right\n1,2,3,3,right\n1,2,3,4,right\n1,2,3,5,right\n1,2,4,1,right\n1,2,4,2,right\n1,2,4,3,right\n1,2,4,4,right\n1,2,4,5,right\n1,2,5,1,right\n1,2,5,2,right\n1,2,5,3,right\n1,2,5,4,right\n1,2,5,5,right\n1,3,1,1,left\n1,3,1,2,left\n1,3,1,3,balanced\n1,3,1,4,right\n1,3,1,5,right\n1,3,2,1,left\n1,3,2,2,right\n1,3,2,3,right\n1,3,2,4,right\n1,3,2,5,right\n1,3,3,1,balanced\n1,3,3,2,right\n1,3,3,3,right\n1,3,3,4,right\n1,3,3,5,right\n1,3,4,1,right\n1,3,4,2,right\n1,3,4,3,right\n1,3,4,4,right\n1,3,4,5,right\n1,3,5,1,right\n1,3,5,2,right\n1,3,5,3,right\n1,3,5,4,right\n1,3,5,5,right\n1,4,1,1,left\n1,4,1,2,left\n1,4,1,3,left\n1,4,1,4,balanced\n1,4,1,5,right\n1,4,2,1,left\n1,4,2,2,balanced\n1,4,2,3,right\n1,4,2,4,right\n1,4,2,5,right\n1,4,3,1,left\n1,4,3,2,right\n1,4,3,3,right\n1,4,3,4,right\n1,4,3,5,right\n1,4,4,1,balanced\n1,4,4,2,right\n1,4,4,3,right\n1,4,4,4,right\n1,4,4,5,right\n1,4,5,1,right\n1,4,5,2,right\n1,4,5,3,right\n1,4,5,4,right\n1,4,5,5,right\n1,5,1,1,left\n1,5,1,2,left\n1,5,1,3,left\n1,5,1,4,left\n1,5,1,5,balanced\n1,5,2,1,left\n1,5,2,2,left\n1,5,2,3,right\n1,5,2,4,right\n1,5,2,5,right\n1,5,3,1,left\n1,5,3,2,right\n1,5,3,3,right\n1,5,3,4,right\n1,5,3,5,right\n1,5,4,1,left\n1,5,4,2,right\n1,5,4,3,right\n1,5,4,4,right\n1,5,4,5,right\n1,5,5,1,balanced\n1,5,5,2,right\n1,5,5,3,right\n1,5,5,4,right\n1,5,5,5,right\n2,1,1,1,left\n2,1,1,2,balanced\n2,1,1,3,right\n2,1,1,4,right\n2,1,1,5,right\n2,1,2,1,balanced\n2,1,2,2,right\n2,1,2,3,right\n2,1,2,4,right\n2,1,2,5,right\n2,1,3,1,right\n2,1,3,2,right\n2,1,3,3,right\n2,1,3,4,right\n2,1,3,5,right\n2,1,4,1,right\n2,1,4,2,right\n2,1,4,3,right\n2,1,4,4,right\n2,1,4,5,right\n2,1,5,1,right\n2,1,5,2,right\n2,1,5,3,right\n2,1,5,4,right\n2,1,5,5,right\n2,2,1,1,left\n2,2,1,2,left\n2,2,1,3,left\n2,2,1,4,balanced\n2,2,1,5,right\n2,2,2,1,left\n2,2,2,2,balanced\n2,2,2,3,right\n2,2,2,4,right\n2,2,2,5,right\n2,2,3,1,left\n2,2,3,2,right\n2,2,3,3,right\n2,2,3,4,right\n2,2,3,5,right\n2,2,4,1,balanced\n2,2,4,2,right\n2,2,4,3,right\n2,2,4,4,right\n2,2,4,5,right\n2,2,5,1,right\n2,2,5,2,right\n2,2,5,3,right\n2,2,5,4,right\n2,2,5,5,right\n2,3,1,1,left\n2,3,1,2,left\n2,3,1,3,left\n2,3,1,4,left\n2,3,1,5,left\n2,3,2,1,left\n2,3,2,2,left\n2,3,2,3,balanced\n2,3,2,4,right\n2,3,2,5,right\n2,3,3,1,left\n2,3,3,2,balanced\n2,3,3,3,right\n2,3,3,4,right\n2,3,3,5,right\n2,3,4,1,left\n2,3,4,2,right\n2,3,4,3,right\n2,3,4,4,right\n2,3,4,5,right\n2,3,5,1,left\n2,3,5,2,right\n2,3,5,3,right\n2,3,5,4,right\n2,3,5,5,right\n2,4,1,1,left\n2,4,1,2,left\n2,4,1,3,left\n2,4,1,4,left\n2,4,1,5,left\n2,4,2,1,left\n2,4,2,2,left\n2,4,2,3,left\n2,4,2,4,balanced\n2,4,2,5,right\n2,4,3,1,left\n2,4,3,2,left\n2,4,3,3,right\n2,4,3,4,right\n2,4,3,5,right\n2,4,4,1,left\n2,4,4,2,balanced\n2,4,4,3,right\n2,4,4,4,right\n2,4,4,5,right\n2,4,5,1,left\n2,4,5,2,right\n2,4,5,3,right\n2,4,5,4,right\n2,4,5,5,right\n2,5,1,1,left\n2,5,1,2,left\n2,5,1,3,left\n2,5,1,4,left\n2,5,1,5,left\n2,5,2,1,left\n2,5,2,2,left\n2,5,2,3,left\n2,5,2,4,left\n2,5,2,5,balanced\n2,5,3,1,left\n2,5,3,2,left\n2,5,3,3,left\n2,5,3,4,right\n2,5,3,5,right\n2,5,4,1,left\n2,5,4,2,left\n2,5,4,3,right\n2,5,4,4,right\n2,5,4,5,right\n2,5,5,1,left\n2,5,5,2,balanced\n2,5,5,3,right\n2,5,5,4,right\n2,5,5,5,right\n3,1,1,1,left\n3,1,1,2,left\n3,1,1,3,balanced\n3,1,1,4,right\n3,1,1,5,right\n3,1,2,1,left\n3,1,2,2,right\n3,1,2,3,right\n3,1,2,4,right\n3,1,2,5,right\n3,1,3,1,balanced\n3,1,3,2,right\n3,1,3,3,right\n3,1,3,4,right\n3,1,3,5,right\n3,1,4,1,right\n3,1,4,2,right\n3,1,4,3,right\n3,1,4,4,right\n3,1,4,5,right\n3,1,5,1,right\n3,1,5,2,right\n3,1,5,3,right\n3,1,5,4,right\n3,1,5,5,right\n3,2,1,1,left\n3,2,1,2,left\n3,2,1,3,left\n3,2,1,4,left\n3,2,1,5,left\n3,2,2,1,left\n3,2,2,2,left\n3,2,2,3,balanced\n3,2,2,4,right\n3,2,2,5,right\n3,2,3,1,left\n3,2,3,2,balanced\n3,2,3,3,right\n3,2,3,4,right\n3,2,3,5,right\n3,2,4,1,left\n3,2,4,2,right\n3,2,4,3,right\n3,2,4,4,right\n3,2,4,5,right\n3,2,5,1,left\n3,2,5,2,right\n3,2,5,3,right\n3,2,5,4,right\n3,2,5,5,right\n3,3,1,1,left\n3,3,1,2,left\n3,3,1,3,left\n3,3,1,4,left\n3,3,1,5,left\n3,3,2,1,left\n3,3,2,2,left\n3,3,2,3,left\n3,3,2,4,left\n3,3,2,5,right\n3,3,3,1,left\n3,3,3,2,left\n3,3,3,3,balanced\n3,3,3,4,right\n3,3,3,5,right\n3,3,4,1,left\n3,3,4,2,left\n3,3,4,3,right\n3,3,4,4,right\n3,3,4,5,right\n3,3,5,1,left\n3,3,5,2,right\n3,3,5,3,right\n3,3,5,4,right\n3,3,5,5,right\n3,4,1,1,left\n3,4,1,2,left\n3,4,1,3,left\n3,4,1,4,left\n3,4,1,5,left\n3,4,2,1,left\n3,4,2,2,left\n3,4,2,3,left\n3,4,2,4,left\n3,4,2,5,left\n3,4,3,1,left\n3,4,3,2,left\n3,4,3,3,left\n3,4,3,4,balanced\n3,4,3,5,right\n3,4,4,1,left\n3,4,4,2,left\n3,4,4,3,balanced\n3,4,4,4,right\n3,4,4,5,right\n3,4,5,1,left\n3,4,5,2,left\n3,4,5,3,right\n3,4,5,4,right\n3,4,5,5,right\n3,5,1,1,left\n3,5,1,2,left\n3,5,1,3,left\n3,5,1,4,left\n3,5,1,5,left\n3,5,2,1,left\n3,5,2,2,left\n3,5,2,3,left\n3,5,2,4,left\n3,5,2,5,left\n3,5,3,1,left\n3,5,3,2,left\n3,5,3,3,left\n3,5,3,4,left\n3,5,3,5,balanced\n3,5,4,1,left\n3,5,4,2,left\n3,5,4,3,left\n3,5,4,4,right\n3,5,4,5,right\n3,5,5,1,left\n3,5,5,2,left\n3,5,5,3,balanced\n3,5,5,4,right\n3,5,5,5,right\n4,1,1,1,left\n4,1,1,2,left\n4,1,1,3,left\n4,1,1,4,balanced\n4,1,1,5,right\n4,1,2,1,left\n4,1,2,2,balanced\n4,1,2,3,right\n4,1,2,4,right\n4,1,2,5,right\n4,1,3,1,left\n4,1,3,2,right\n4,1,3,3,right\n4,1,3,4,right\n4,1,3,5,right\n4,1,4,1,balanced\n4,1,4,2,right\n4,1,4,3,right\n4,1,4,4,right\n4,1,4,5,right\n4,1,5,1,right\n4,1,5,2,right\n4,1,5,3,right\n4,1,5,4,right\n4,1,5,5,right\n4,2,1,1,left\n4,2,1,2,left\n4,2,1,3,left\n4,2,1,4,left\n4,2,1,5,left\n4,2,2,1,left\n4,2,2,2,left\n4,2,2,3,left\n4,2,2,4,balanced\n4,2,2,5,right\n4,2,3,1,left\n4,2,3,2,left\n4,2,3,3,right\n4,2,3,4,right\n4,2,3,5,right\n4,2,4,1,left\n4,2,4,2,balanced\n4,2,4,3,right\n4,2,4,4,right\n4,2,4,5,right\n4,2,5,1,left\n4,2,5,2,right\n4,2,5,3,right\n4,2,5,4,right\n4,2,5,5,right\n4,3,1,1,left\n4,3,1,2,left\n4,3,1,3,left\n4,3,1,4,left\n4,3,1,5,left\n4,3,2,1,left\n4,3,2,2,left\n4,3,2,3,left\n4,3,2,4,left\n4,3,2,5,left\n4,3,3,1,left\n4,3,3,2,left\n4,3,3,3,left\n4,3,3,4,balanced\n4,3,3,5,right\n4,3,4,1,left\n4,3,4,2,left\n4,3,4,3,balanced\n4,3,4,4,right\n4,3,4,5,right\n4,3,5,1,left\n4,3,5,2,left\n4,3,5,3,right\n4,3,5,4,right\n4,3,5,5,right\n4,4,1,1,left\n4,4,1,2,left\n4,4,1,3,left\n4,4,1,4,left\n4,4,1,5,left\n4,4,2,1,left\n4,4,2,2,left\n4,4,2,3,left\n4,4,2,4,left\n4,4,2,5,left\n4,4,3,1,left\n4,4,3,2,left\n4,4,3,3,left\n4,4,3,4,left\n4,4,3,5,left\n4,4,4,1,left\n4,4,4,2,left\n4,4,4,3,left\n4,4,4,4,balanced\n4,4,4,5,right\n4,4,5,1,left\n4,4,5,2,left\n4,4,5,3,left\n4,4,5,4,right\n4,4,5,5,right\n4,5,1,1,left\n4,5,1,2,left\n4,5,1,3,left\n4,5,1,4,left\n4,5,1,5,left\n4,5,2,1,left\n4,5,2,2,left\n4,5,2,3,left\n4,5,2,4,left\n4,5,2,5,left\n4,5,3,1,left\n4,5,3,2,left\n4,5,3,3,left\n4,5,3,4,left\n4,5,3,5,left\n4,5,4,1,left\n4,5,4,2,left\n4,5,4,3,left\n4,5,4,4,left\n4,5,4,5,balanced\n4,5,5,1,left\n4,5,5,2,left\n4,5,5,3,left\n4,5,5,4,balanced\n4,5,5,5,right\n5,1,1,1,left\n5,1,1,2,left\n5,1,1,3,left\n5,1,1,4,left\n5,1,1,5,balanced\n5,1,2,1,left\n5,1,2,2,left\n5,1,2,3,right\n5,1,2,4,right\n5,1,2,5,right\n5,1,3,1,left\n5,1,3,2,right\n5,1,3,3,right\n5,1,3,4,right\n5,1,3,5,right\n5,1,4,1,left\n5,1,4,2,right\n5,1,4,3,right\n5,1,4,4,right\n5,1,4,5,right\n5,1,5,1,balanced\n5,1,5,2,right\n5,1,5,3,right\n5,1,5,4,right\n5,1,5,5,right\n5,2,1,1,left\n5,2,1,2,left\n5,2,1,3,left\n5,2,1,4,left\n5,2,1,5,left\n5,2,2,1,left\n5,2,2,2,left\n5,2,2,3,left\n5,2,2,4,left\n5,2,2,5,balanced\n5,2,3,1,left\n5,2,3,2,left\n5,2,3,3,left\n5,2,3,4,right\n5,2,3,5,right\n5,2,4,1,left\n5,2,4,2,left\n5,2,4,3,right\n5,2,4,4,right\n5,2,4,5,right\n5,2,5,1,left\n5,2,5,2,balanced\n5,2,5,3,right\n5,2,5,4,right\n5,2,5,5,right\n5,3,1,1,left\n5,3,1,2,left\n5,3,1,3,left\n5,3,1,4,left\n5,3,1,5,left\n5,3,2,1,left\n5,3,2,2,left\n5,3,2,3,left\n5,3,2,4,left\n5,3,2,5,left\n5,3,3,1,left\n5,3,3,2,left\n5,3,3,3,left\n5,3,3,4,left\n5,3,3,5,balanced\n5,3,4,1,left\n5,3,4,2,left\n5,3,4,3,left\n5,3,4,4,right\n5,3,4,5,right\n5,3,5,1,left\n5,3,5,2,left\n5,3,5,3,balanced\n5,3,5,4,right\n5,3,5,5,right\n5,4,1,1,left\n5,4,1,2,left\n5,4,1,3,left\n5,4,1,4,left\n5,4,1,5,left\n5,4,2,1,left\n5,4,2,2,left\n5,4,2,3,left\n5,4,2,4,left\n5,4,2,5,left\n5,4,3,1,left\n5,4,3,2,left\n5,4,3,3,left\n5,4,3,4,left\n5,4,3,5,left\n5,4,4,1,left\n5,4,4,2,left\n5,4,4,3,left\n5,4,4,4,left\n5,4,4,5,balanced\n5,4,5,1,left\n5,4,5,2,left\n5,4,5,3,left\n5,4,5,4,balanced\n5,4,5,5,right\n5,5,1,1,left\n5,5,1,2,left\n5,5,1,3,left\n5,5,1,4,left\n5,5,1,5,left\n5,5,2,1,left\n5,5,2,2,left\n5,5,2,3,left\n5,5,2,4,left\n5,5,2,5,left\n5,5,3,1,left\n5,5,3,2,left\n5,5,3,3,left\n5,5,3,4,left\n5,5,3,5,left\n5,5,4,1,left\n5,5,4,2,left\n5,5,4,3,left\n5,5,4,4,left\n5,5,4,5,left\n5,5,5,1,left\n5,5,5,2,left\n5,5,5,3,left\n5,5,5,4,left\n5,5,5,5,balanced\n', 'llmfe/buffer.py': '"""A multi-island experience buffer that implements the evolutionary algorithm."""\nfrom __future__ import annotations\n\nimport profile\nfrom collections.abc import Mapping, Sequence\nimport copy\nimport dataclasses\nimport time\nfrom typing import Any, Tuple, Mapping\n\nfrom absl import logging\nimport numpy as np\nimport random\nimport os\nimport pandas as pd\nimport scipy\nimport sys\n\nfrom llmfe import code_manipulation\nfrom llmfe import config as config_lib\nsys.path.append(\'..\')\nfrom utils import serialize\nfrom utils import is_categorical\nSignature = Tuple[float, ...]\nScoresPerTest = Mapping[Any, float]\n\n\ndef _softmax(logits: np.ndarray, temperature: float) -> np.ndarray:\n    """Returns the tempered softmax of 1D finite `logits`."""\n    if not np.all(np.isfinite(logits)):\n        non_finites = set(logits[~np.isfinite(logits)])\n        raise ValueError(f\'`logits` contains non-finite value(s): {non_finites}\')\n    if not np.issubdtype(logits.dtype, np.floating):\n        logits = np.array(logits, dtype=np.float32)\n\n    result = scipy.special.softmax(logits / temperature, axis=-1)\n    index = np.argmax(result)\n    result[index] = 1 - np.sum(result[0:index]) - np.sum(result[index + 1:])\n    return result\n\ndef _reduce_score(scores_per_test: ScoresPerTest) -> float:\n    test_scores = [scores_per_test[k] for k in scores_per_test.keys()]\n    return sum(test_scores) / len(test_scores)\n\n\ndef _get_signature(scores_per_test: ScoresPerTest) -> Signature:\n    """Represents test scores as a canonical signature."""\n    return tuple(scores_per_test[k] for k in sorted(scores_per_test.keys()))\n\n\n@dataclasses.dataclass(frozen=True)\nclass Prompt:\n    """ A prompt produced by the Experience Buffer, to be sent to Samplers.\n\n    Args:\n      code: The prompt, ending with the header of the function to be completed.\n      version_generated: The function to be completed is `_v{version_generated}`.\n      island_id: Identifier of the island that produced the samples\n                included in the prompt. Used to direct the newly generated sample\n                into the same island.\n    """\n    code: str\n    version_generated: int\n    island_id: int\n    data_input: pd.DataFrame\n    data_output: list\n\n\nclass ExperienceBuffer:\n    """A collection of programs, organized as islands."""\n\n    def __init__(\n            self,\n            config: config_lib.ExperienceBufferConfig,\n            template: code_manipulation.Program,\n            function_to_evolve: str,\n            meta_data: dict\n    ) -> None:\n        self._config: config_lib.ExperienceBufferConfig = config\n        self._template: code_manipulation.Program = template\n        self._function_to_evolve: str = function_to_evolve\n        self._meta_data: dict = meta_data\n\n        # Initialize empty islands.\n        self._islands: list[Island] = []\n        for _ in range(config.num_islands):\n            self._islands.append(\n                Island(template, function_to_evolve, config.functions_per_prompt, \n                       meta_data,\n                       config.cluster_sampling_temperature_init,\n                       config.cluster_sampling_temperature_period))\n        self._best_score_per_island: list[float] = (\n                [-float(\'inf\')] * config.num_islands)\n        self._best_program_per_island: list[code_manipulation.Function | None] = (\n                [None] * config.num_islands)\n        self._best_scores_per_test_per_island: list[ScoresPerTest | None] = (\n                [None] * config.num_islands)\n\n        self._last_reset_time: float = time.time()\n\n\n    def get_prompt(self) -> Prompt:\n        """Returns a prompt containing samples from one chosen island."""\n        island_id = np.random.randint(len(self._islands))\n        code, version_generated, data_input, data_output = self._islands[island_id].get_prompt()\n        \n        return Prompt(code, version_generated, island_id, data_input, data_output)\n\n\n    def _register_program_in_island(\n            self,\n            program: code_manipulation.Function,\n            input_data: pd.DataFrame,\n            output_data: pd.DataFrame,\n            island_id: int,\n            scores_per_test: ScoresPerTest,\n            **kwargs\n    ) -> None:\n        """Registers `program` in the specified island."""\n        self._islands[island_id].register_program(input_data, output_data, program, scores_per_test)\n        score = _reduce_score(scores_per_test)\n        if score > self._best_score_per_island[island_id]:\n            self._best_program_per_island[island_id] = program\n            self._best_scores_per_test_per_island[island_id] = scores_per_test\n            self._best_score_per_island[island_id] = score\n            logging.info(\'Best score of island %d increased to %s\', island_id, score)\n\n        profiler: profile.Profiler = kwargs.get(\'profiler\', None)\n        if profiler:\n            global_sample_nums = kwargs.get(\'global_sample_nums\', None)\n            sample_time = kwargs.get(\'sample_time\', None)\n            evaluate_time = kwargs.get(\'evaluate_time\', None)\n            program.data_input = input_data\n            program.data_output = output_data\n            program.score = score\n            program.global_sample_nums = global_sample_nums\n            program.sample_time = sample_time\n            program.evaluate_time = evaluate_time\n            profiler.register_function(program)\n\n\n    def register_program(\n            self,\n            program: code_manipulation.Function,\n            island_id: int | None,\n            scores_per_test: ScoresPerTest,\n            input_data: pd.DataFrame,\n            output_data: pd.DataFrame,\n            **kwargs\n    ) -> None:\n        """Registers new `program` skeleton hypotheses in the experience buffer."""\n        if island_id is None:\n            for island_id in range(len(self._islands)):\n                self._register_program_in_island(program, input_data, output_data, island_id, scores_per_test, **kwargs)\n        else:\n            self._register_program_in_island(program, input_data, output_data, island_id, scores_per_test, **kwargs)\n\n        # Check island reset\n        if time.time() - self._last_reset_time > self._config.reset_period:\n            self._last_reset_time = time.time()\n            self.reset_islands()\n\n\n    def reset_islands(self) -> None:\n        """Resets the weaker half of islands."""\n        # Sort best scores after adding minor noise to break ties.\n        indices_sorted_by_score: np.ndarray = np.argsort(\n            self._best_score_per_island +\n            np.random.randn(len(self._best_score_per_island)) * 1e-6)\n        num_islands_to_reset = self._config.num_islands // 2\n        reset_islands_ids = indices_sorted_by_score[:num_islands_to_reset]\n        keep_islands_ids = indices_sorted_by_score[num_islands_to_reset:]\n        for island_id in reset_islands_ids:\n            self._islands[island_id] = Island(\n                self._template,\n                self._function_to_evolve,\n                self._config.functions_per_prompt,\n                self._meta_data,\n                self._config.cluster_sampling_temperature_init,\n                self._config.cluster_sampling_temperature_period)\n            self._best_score_per_island[island_id] = -float(\'inf\')\n            founder_island_id = np.random.choice(keep_islands_ids)\n            founder = self._best_program_per_island[founder_island_id]\n            founder_scores = self._best_scores_per_test_per_island[founder_island_id]\n            self._register_program_in_island(founder, None, None, island_id, founder_scores)\n\n\nclass Island:\n    """A sub-population of the program skeleton experience buffer."""\n\n    def __init__(\n            self,\n            template: code_manipulation.Program,\n            function_to_evolve: str,\n            functions_per_prompt: int,\n            meta_data: dict,\n            cluster_sampling_temperature_init: float,\n            cluster_sampling_temperature_period: int,\n    ) -> None:\n        self._template: code_manipulation.Program = template\n        self._function_to_evolve: str = function_to_evolve\n        self._functions_per_prompt: int = functions_per_prompt\n        self._meta_data: dict = meta_data\n        self._cluster_sampling_temperature_init = cluster_sampling_temperature_init\n        self._cluster_sampling_temperature_period = (\n            cluster_sampling_temperature_period)\n\n        self._clusters: dict[Signature, Cluster] = {}\n        self._num_programs: int = 0\n\n\n    def register_program(\n            self,\n            data_input: pd.DataFrame,\n            data_output: list,\n            program: code_manipulation.Function,\n            scores_per_test: ScoresPerTest,\n    ) -> None:\n        """Stores a program on this island, in its appropriate cluster."""\n        signature = _get_signature(scores_per_test)\n        if signature not in self._clusters:\n            score = _reduce_score(scores_per_test)\n            self._clusters[signature] = Cluster(score, program, data_input, data_output)\n        else:\n            self._clusters[signature].register_program(program, data_input, data_output)\n        self._num_programs += 1\n\n\n    def get_prompt(self) -> tuple[str, int]:\n        """Constructs a prompt containing equation program skeletons from this island."""\n        signatures = list(self._clusters.keys())\n        cluster_scores = np.array(\n            [self._clusters[signature].score for signature in signatures])\n        \n        period = self._cluster_sampling_temperature_period\n        temperature = self._cluster_sampling_temperature_init * (\n                1 - (self._num_programs % period) / period)\n        probabilities = _softmax(cluster_scores, temperature)\n\n        functions_per_prompt = min(len(self._clusters), self._functions_per_prompt)\n\n        idx = np.random.choice(\n            len(signatures), size=functions_per_prompt, p=probabilities)\n        chosen_signatures = [signatures[i] for i in idx]\n        implementations = []\n        scores = []\n        for signature in chosen_signatures:\n            cluster = self._clusters[signature]\n            implementations.append(cluster.sample_program())\n            scores.append(cluster.score)\n\n        indices = np.argsort(scores)\n        sorted_implementations = [implementations[i] for i in indices]\n        version_generated = len(sorted_implementations) + 1\n        return self._generate_prompt(sorted_implementations), version_generated, sorted_implementations[-1].data_input, sorted_implementations[-1].data_output\n\n\n    def _generate_prompt(\n            self,\n            implementations: Sequence[code_manipulation.Function]) -> str:\n        """ Create a prompt containing a sequence of function `implementations`."""\n        implementations = copy.deepcopy(implementations)\n        # Format the names and docstrings of functions to be included in the prompt.\n        versioned_functions: list[code_manipulation.Function] = []\n        input_data = []\n        output_data = []\n        for i, implementation in enumerate(implementations):\n            new_function_name = f\'{self._function_to_evolve}_v{i}\'\n            implementation.name = new_function_name\n            # Update the docstring for all subsequent functions after `_v0`.\n            if i >= 1:\n                implementation.docstring = (\n                    f\'Improved version of `{self._function_to_evolve}_v{i - 1}`.\'\n                    )\n            # If the function is recursive, replace calls to itself with its new name.\n            input_data.append(implementation.data_input)\n            output_data.append(implementation.data_output)\n            implementation = code_manipulation.rename_function_calls(\n                str(implementation), self._function_to_evolve, new_function_name)\n            \n            versioned_functions.append(\n                code_manipulation.text_to_function(implementation))\n\n        # Create header of new function to be completed\n        next_version = len(implementations)\n        new_function_name = f\'{self._function_to_evolve}_v{next_version}\'\n        header = dataclasses.replace(\n            implementations[-1],\n            name=new_function_name,\n            body=\'\',\n            docstring=(\'Improved version of \'\n                       f\'`{self._function_to_evolve}_v{next_version - 1}`. Think and suggest new features.\'),\n        )\n        versioned_functions.append(header)\n        # Replace functions in the template with the list constructed here.\n        prompt = dataclasses.replace(self._template, functions=versioned_functions)\n\n        in_context_desc = ""\n        df_input = input_data[-1].copy()\n        if not df_input.index.name:\n            df_input.reset_index(inplace=True, drop=True)\n        if any(col.startswith(\'Unnamed:\') for col in df_input.columns):\n            # Drop the \'Unnamed:\' column(s)\n            df_input = df_input.loc[:, ~df_input.columns.str.startswith(\'Unnamed:\')]\n        \n        df_output = pd.DataFrame(output_data[-1], columns=[\'Result\'])\n        df_current = df_input.join(df_output)\n        df_current = df_current.sample(frac=1).head(10)\n        \n        total_column_list = [df_current.columns.tolist()]\n        categorical_indicator = [is_categorical(df_current.iloc[:, i]) for i in range(df_current.shape[1])]\n        \n        for selected_column in total_column_list:\n            for icl_idx, icl_row in df_current.iterrows():\n                icl_row = icl_row[selected_column]\n                in_context_desc += serialize(icl_row)\n                in_context_desc += "\\n"\n\n            feature_name_list = []\n            sel_cat_idx = [df_current.columns.tolist().index(col_name) for col_name in selected_column]\n            is_cat_sel = np.array(categorical_indicator)[sel_cat_idx]\n            \n            for cidx, cname in enumerate(selected_column):\n                if cname == "Result":\n                    break\n                if is_cat_sel[cidx] == True:\n                    clist = df_current[cname].unique().tolist()\n                    clist = [str(c) for c in clist]\n\n                    clist_str = ", ".join(clist)\n                    desc = self._meta_data[cname] if cname in self._meta_data.keys() else ""\n                    feature_name_list.append(f"- {cname}: {desc} (categorical variable with categories [{clist_str}])")\n                else:\n                    min_val, max_val = input_data[-1][cname].min(), input_data[-1][cname].max()\n                    desc = self._meta_data[cname] if cname in self._meta_data.keys() else cname.replace(\'_\', \' \')\n                    feature_name_list.append(f"- {cname}: {desc} (numerical variable within range [{min_val}, {max_val}])")\n            \n            feature_desc = "\\n".join(feature_name_list)\n\n        new_prompt = str(prompt)\n        random_number = random.randint(0,2)\n\n        if random_number == 0:\n            template_prefix = os.path.join("prompts", "operations_head.txt")\n            if os.path.exists(template_prefix):\n                with open(template_prefix) as f:\n                    prefix = f.read()\n            template_suffix = os.path.join("prompts", "operations_tail.txt")\n            if os.path.exists(template_suffix):\n                with open(template_suffix) as f:\n                    suffix = f.read()\n            new_prompt = new_prompt.replace(\'[PREFIX]\', prefix).replace(\'[SUFFIX]\', suffix)\n        else:\n            template_prefix = os.path.join("prompts", "domain_head.txt")\n            if os.path.exists(template_prefix):\n                with open(template_prefix) as f:\n                    prefix = f.read()\n            template_suffix = os.path.join("prompts", "domain_tail.txt")\n            if os.path.exists(template_suffix):\n                with open(template_suffix) as f:\n                    suffix = f.read()\n            new_prompt = new_prompt.replace(\'[PREFIX]\', prefix).replace(\'[SUFFIX]\', suffix)\n        new_prompt = new_prompt.replace(\'[EXAMPLES]\', in_context_desc).replace(\'[FEATURES]\', feature_desc)\n\n        return new_prompt\n\n\nclass Cluster:\n    """ A cluster of programs on the same island and with the same Signature. """\n\n    def __init__(self, score: float, implementation: code_manipulation.Function, data_input: pd.DataFrame, data_output: pd.DataFrame):\n        self._score = score\n        self._programs: list[code_manipulation.Function] = [implementation]\n        self._lengths: list[int] = [len(str(implementation))]\n        self._data_input : list[pd.DataFrame] = [data_input]\n        self._data_output : list[array] = [data_output]\n\n    @property\n    def score(self) -> float:\n        return self._score\n\n    def register_program(self, program: code_manipulation.Function, data_input: pd.DataFrame, data_output: pd.DataFrame) -> None:\n        """Adds `program` to the cluster."""\n        self._programs.append(program)\n        self._lengths.append(len(str(program)))\n        self._data_input.append(data_input)\n        self._data_output.append(data_output)\n\n    def sample_program(self) -> code_manipulation.Function:\n        """Samples a program, giving higher probability to shorther programs."""\n        normalized_lengths = (np.array(self._lengths) - min(self._lengths)) / (\n                max(self._lengths) + 1e-6)\n        probabilities = _softmax(-normalized_lengths, temperature=1.0)\n\n        return np.random.choice(self._programs, p=probabilities)\n', 'llmfe/sampler.py': '""" Class for sampling new programs. """\nfrom __future__ import annotations\nfrom abc import ABC, abstractmethod\n\nfrom typing import Collection, Sequence, Type\nimport numpy as np\nimport time\n\nfrom llmfe import evaluator\nfrom llmfe import buffer\nfrom llmfe import config as config_lib\nimport requests\nimport json\nimport http.client\nimport os\n\n\n\nclass LLM(ABC):\n    def __init__(self, samples_per_prompt: int) -> None:\n        self._samples_per_prompt = samples_per_prompt\n\n    def _draw_sample(self, prompt: str) -> str:\n        """ Return a predicted continuation of `prompt`."""\n        raise NotImplementedError(\'Must provide a language model.\')\n\n    @abstractmethod\n    def draw_samples(self, prompt: str) -> Collection[str]:\n        """ Return multiple predicted continuations of `prompt`. """\n        return [self._draw_sample(prompt) for _ in range(self._samples_per_prompt)]\n\n\n\nclass Sampler:\n    """ Node that samples program skeleton continuations and sends them for analysis. """\n    _global_samples_nums: int = 1\n\n    def __init__(\n            self,\n            database: buffer.ExperienceBuffer,\n            evaluators: Sequence[evaluator.Evaluator],\n            samples_per_prompt: int,\n            meta_data: dict,\n            config: config_lib.Config,\n            max_sample_nums: int | None = None,\n            llm_class: Type[LLM] = LLM,\n    ):\n        self._samples_per_prompt = samples_per_prompt\n        self._database = database\n        self._evaluators = evaluators\n        self._meta_data = meta_data\n        self._llm = llm_class(samples_per_prompt)\n        self._max_sample_nums = max_sample_nums\n        self.config = config\n        self.__class__._global_samples_nums = 1\n\n    \n    def sample(self, **kwargs):\n        """ Continuously gets prompts, samples programs, sends them for analysis. """\n        while True:\n            # stop the search process if hit global max sample nums\n            if self._max_sample_nums//5 and self.__class__._global_samples_nums >= self._max_sample_nums//5:\n                break\n            \n            prompt = self._database.get_prompt()\n            reset_time = time.time()\n            samples = self._llm.draw_samples(prompt.code,self.config)\n            sample_time = (time.time() - reset_time) / self._samples_per_prompt\n\n            # This loop can be executed in parallel on remote evaluator machines.\n            for sample in samples:\n                sample = "\\n    import pandas as pd\\n    import numpy as np\\n" + sample\n                self._global_sample_nums_plus_one()\n                cur_global_sample_nums = self._get_global_sample_nums()\n                chosen_evaluator: evaluator.Evaluator = np.random.choice(self._evaluators)\n                chosen_evaluator.analyse(\n                    sample,\n                    prompt.island_id,\n                    prompt.data_input,\n                    prompt.data_output,\n                    prompt.version_generated,\n                    **kwargs,\n                    global_sample_nums=cur_global_sample_nums,\n                    sample_time=sample_time\n                )\n\n    def _get_global_sample_nums(self) -> int:\n        return self.__class__._global_samples_nums\n\n    def set_global_sample_nums(self, num):\n        self.__class__._global_samples_nums = num\n\n    def _global_sample_nums_plus_one(self):\n        self.__class__._global_samples_nums += 1\n\n\n\ndef _extract_body(sample: str, config: config_lib.Config) -> str:\n    """\n    Extract the function body from a response sample, removing any preceding descriptions\n    and the function signature. Preserves indentation.\n    ------------------------------------------------------------------------------------------------------------------\n    Input example:\n    ```\n    This is a description...\n    def function_name(...):\n        return ...\n    Additional comments...\n    ```\n    ------------------------------------------------------------------------------------------------------------------\n    Output example:\n    ```\n        return ...\n    Additional comments...\n    ```\n    ------------------------------------------------------------------------------------------------------------------\n    If no function definition is found, returns the original sample.\n    """\n    lines = sample.splitlines()\n    func_body_lineno = 0\n    find_def_declaration = False\n    \n    for lineno, line in enumerate(lines):\n        # find the first \'def\' program statement in the response\n        if line[:3] == \'def\':\n            func_body_lineno = lineno\n            find_def_declaration = True\n            break\n    \n    if find_def_declaration:\n        # for gpt APIs\n        if config.use_api:\n            code = \'\'\n            for line in lines[func_body_lineno + 1:]:\n                code += line + \'\\n\'\n        \n        # for mixtral\n        else:\n            code = \'\'\n            indent = \'    \'\n            for line in lines[func_body_lineno + 1:]:\n                if line[:4] != indent:\n                    line = indent + line\n                code += line + \'\\n\'\n        \n        return code\n    \n    return sample\n\n\n\nclass LocalLLM(LLM):\n    def __init__(self, samples_per_prompt: int, batch_inference: bool = True, trim=True) -> None:\n        """\n        Args:\n            batch_inference: Use batch inference when sample equation program skeletons. The batch size equals to the samples_per_prompt.\n        """\n        super().__init__(samples_per_prompt)\n\n        url = "http://127.0.0.1:5000/completions"\n        instruction_prompt = ("You are a helpful assistant tasked with discovering new features/ dropping less important feaures for the given prediction task. \\\n                             Complete the \'modify_features\' function below, considering the physical meaning and relationships of inputs.\\n\\n")\n        self._batch_inference = batch_inference\n        self._url = url\n        self._instruction_prompt = instruction_prompt\n        self._trim = trim\n\n\n    def draw_samples(self, prompt: str, config: config_lib.Config) -> Collection[str]:\n        """Returns multiple equation program skeleton hypotheses for the given `prompt`."""\n        if config.use_api:\n            return self._draw_samples_api(prompt, config)\n        else:\n            return self._draw_samples_local(prompt, config)\n\n\n    def _draw_samples_local(self, prompt: str, config: config_lib.Config) -> Collection[str]:    \n        # instruction\n        prompt = \'\\n\'.join([self._instruction_prompt, prompt])\n        while True:\n            try:\n                all_samples = []\n                # response from llm server\n                if self._batch_inference:\n                    response = self._do_request(prompt)\n                    for res in response:\n                        all_samples.append(res)\n                else:\n                    for _ in range(self._samples_per_prompt):\n                        response = self._do_request(prompt)\n                        all_samples.append(response)\n\n                # trim equation program skeleton body from samples\n                if self._trim:\n                    all_samples = [_extract_body(sample, config) for sample in all_samples]\n                \n                return all_samples\n            except Exception:\n                continue\n\n\n    def _draw_samples_api(self, prompt: str, config: config_lib.Config) -> Collection[str]:\n        all_samples = []\n        prompt = \'\\n\'.join([self._instruction_prompt, prompt])\n        \n        for _ in range(self._samples_per_prompt):\n            while True:\n                try:\n                    conn = http.client.HTTPSConnection("api.openai.com")\n                    payload = json.dumps({\n                        "max_tokens": 512,\n                        "model": config.api_model,\n                        "messages": [\n                            {\n                                "role": "user",\n                                "content": prompt\n                            }\n                        ]\n                    })\n                    headers = {\n                        \'Authorization\': f"Bearer {os.environ[\'API_KEY\']}",\n                        \'User-Agent\': \'Apifox/1.0.0 (https://apifox.com)\',\n                        \'Content-Type\': \'application/json\'\n                    }\n                    conn.request("POST", "/v1/chat/completions", payload, headers)\n                    res = conn.getresponse()\n                    data = json.loads(res.read().decode("utf-8"))\n                    response = data[\'choices\'][0][\'message\'][\'content\']\n                    \n                    if self._trim:\n                        response = _extract_body(response, config)\n                    \n                    all_samples.append(response)\n                    break\n\n                except Exception:\n                    continue\n        \n        return all_samples\n    \n    \n    def _do_request(self, content: str) -> str:\n        content = content.strip(\'\\n\').strip()\n        # repeat the prompt for batch inference\n        repeat_prompt: int = self._samples_per_prompt if self._batch_inference else 1\n        \n        data = {\n            \'prompt\': content,\n            \'repeat_prompt\': repeat_prompt,\n            \'params\': {\n                \'do_sample\': True,\n                \'temperature\': None,\n                \'top_k\': None,\n                \'top_p\': None,\n                \'add_special_tokens\': False,\n                \'skip_special_tokens\': True,\n            }\n        }\n        \n        headers = {\'Content-Type\': \'application/json\'}\n        response = requests.post(self._url, data=json.dumps(data), headers=headers)\n        \n        if response.status_code == 200: #Server status code 200 indicates successful HTTP request! \n            response = response.json()["content"]\n            \n            return response if self._batch_inference else response[0]\n\n', 'prompts/domain_head.txt': '###\n<Role>  \nYou are a data scientist expert in the field of the given dataset.  \nYour role is to apply your domain expertise to identify and create, and refine the most informative features that solve the <Task> effectively.  \n\n###  \n<Instructions>  \n  - You are provided with the task description, a list of existing features, and data examples.  \n  - Use your domain knowledge to derive features that capture meaningful patterns, trends, or relationships inherent in the data.\n  - Prioritize features that have high potential to enhance the model’s ability to solve the <Task>, considering both relevance and predictive power.  \n  - For each derived feature, provide:  \n    - A clear explanation of how it was derived and justification of its relevance for solving the <Task>.  \n  - Ensure your approach remains grounded in the context of the dataset and the <Task>, and aim for features that are interpretable and actionable.', 'prompts/domain_tail.txt': 'Please generate as many new features as possible using the information from the task, feature descriptions, examples, and your domain understanding of the dataset.  \nRemove any irrelevant, redundant, or less informative features to enhance overall performance.  \n\nFirst, describe your new feature transformation and the main steps in a concise, one-sentence docstring.  \nThen, implement it in Python as a function that adheres to the given specifications.  \n\nAvoid providing any further explanations or additional descriptions.', 'prompts/operations_head.txt': '###\n<Role>\nYou are a data scientist with expert knowledge about the provided dataset.\nYour primary responsibility is to identify the most informative features that can enhance the solution to the specified <Task>.\n\n###\n<Instructions>\n  - You are given a task description, a list of existing features, a set of advanced operators, and sample data.\n  - Your objective is to leverage the provided advanced operators within <Operators> to generate meaningful and insightful features that enhance task performance. These operators have been carefully curated to extract deeper patterns from the data.\n  - Avoid relying on basic arithmetic operators (e.g., addition, subtraction, multiplication, or division). Instead, focus exclusively on the provided advanced operators inside <Operators>.\n  - For each feature you derive, provide a concise explanation of why it is relevant and to solving the <Task> in the docstring.\n\n###\n<Operators>\n  - General Operators: Frequency (Frequency of feature in the data)\n  - Numerical Input Operators: Absolute, Logarithm, Square Root, Sigmoid, Square, Round, Residual\n  - Numeric-Numeric Operators: Minimum, Maximum\n  - Categorical-Numeric Operators: GroupByThenMin, GroupByThenMax, GroupByThenMean, GroupByThenMedian, GroupByThenStd, GroupByThenRank\n  - Categorical-Categorical Operators: Combine, CombineThenFreq, GroupByThenNUnique', 'prompts/operations_tail.txt': "Please generate as many new features strictly using the operations mentioned in <Operators>, the information from the task, feature description, examples, and your own understanding of the dataset and it's domain.\nEnhance the model performance by dropping irrelevant, redundant or less informative features, that do not contribute toward the <Task>.\n\nFirst, describe your new feature transformation and main steps in one sentence. The description must be inside the docstring.\nNext, implement it in Python as a function that matches the specification.\n\nAvoid providing any additional description.", 'specs/specification_balance-scale.txt': '"""\n[PREFIX]\n\n###\n<Task>\nWhich direction does the balance scale tip to? Right, left, or balanced?\n\n###\n<Features>\n[FEATURES]\n\n###\n<Examples>\n[EXAMPLES]\n[SUFFIX]\n"""\n\n@evaluate.run\ndef evaluate(data: dict):\n    """ Evaluate the feature transformations on data observations."""\n\n    from sklearn import preprocessing\n    from sklearn.model_selection import StratifiedKFold, KFold\n    from sklearn.metrics import accuracy_score\n    from preprocessing import preprocess_datasets\n    import xgboost as xgb\n    import torch\n    import numpy as np\n\n    # Load data observations\n    label_encoder = preprocessing.LabelEncoder()\n    inputs, outputs, is_cat, is_regression = data[\'inputs\'], data[\'outputs\'], data[\'is_cat\'], data[\'is_regression\']\n    X = modify_features(inputs)\n    y = label_encoder.fit_transform(outputs)\n\n    # Encode categorical string columns\n    for col in X.columns:\n        if X[col].dtype == \'string\':\n            X[col] = label_encoder.fit_transform(X[col])\n\n    # Choose cross-validation strategy\n    kf = KFold(n_splits=4, shuffle=True, random_state=42) if is_regression else StratifiedKFold(n_splits=4, shuffle=True, random_state=42)\n\n    scores = []\n\n    # 4-Fold Cross-Validation\n    for train_idx, test_idx in kf.split(X, y):\n        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]\n        y_train, y_test = y[train_idx], y[test_idx]\n\n        X_train_new, X_test_new = preprocess_datasets(X_train, X_test, None)\n        \n        # Convert to tensors\n        X_train = torch.tensor(X_train_new.to_numpy())\n        X_test = torch.tensor(X_test_new.to_numpy())\n\n        # Load model\n        model = xgb.XGBClassifier(random_state=42)\n\n        model.fit(X_train_new, y_train)\n        \n        y_pred = model.predict(X_test_new)\n        score = accuracy_score(y_test, y_pred)\n        scores.append(score)\n\n    # Calculate average score\n    avg_score = np.mean(scores)\n\n    return avg_score, inputs, outputs\n\n@equation.evolve\ndef modify_features(df_input) -> pd.DataFrame:\n    """\n    Thought 1: The absolute difference between Left-Weight and Right-Weight can capture the imbalance in weight distribution.\n    New Feature 1: weight_difference | weight_difference = abs(Left-Weight - Right-Weight)\n    """\n    df_output = df_input.copy()\n    \n    # Calculate absolute difference between Left-Weight and Right-Weight\n    df_output[\'weight_difference\'] = abs(df_output[\'Left-Weight\'] - df_output[\'Right-Weight\'])\n    \n    return df_output'}
UPSTREAM_MANIFEST = {'repository': 'https://github.com/nikhilsab/LLMFE', 'revision': 'e9a58ca3416ae74dcc7549dffdf79b9289604091', 'files': {'LICENSE': {'url': 'https://raw.githubusercontent.com/nikhilsab/LLMFE/e9a58ca3416ae74dcc7549dffdf79b9289604091/LICENSE', 'sha256': 'b72e12daf86bcd663836cf05646cf18a89f22dbccee3fc0a862cd14edde4db08'}, 'specs/specification_balance-scale.txt': {'url': 'https://raw.githubusercontent.com/nikhilsab/LLMFE/e9a58ca3416ae74dcc7549dffdf79b9289604091/specs/specification_balance-scale.txt', 'sha256': 'f7ef8c5a554c5792d607e0d3eb9da8e5df1aad7d9c8057082bde032220e66ac9'}, 'data/balance-scale.csv': {'url': 'https://raw.githubusercontent.com/nikhilsab/LLMFE/e9a58ca3416ae74dcc7549dffdf79b9289604091/data/balance-scale.csv', 'sha256': '5b72a5574a7ae36fa2f4343bf656f9f4a3e99af3f936940d9fa4b5dfbf1848a4'}, 'llmfe/buffer.py': {'url': 'https://raw.githubusercontent.com/nikhilsab/LLMFE/e9a58ca3416ae74dcc7549dffdf79b9289604091/llmfe/buffer.py', 'sha256': '1034e9896ab926695c005b2e24b4639d80590cb6e79df3f35524d5fa5ae06b8b'}, 'llmfe/sampler.py': {'url': 'https://raw.githubusercontent.com/nikhilsab/LLMFE/e9a58ca3416ae74dcc7549dffdf79b9289604091/llmfe/sampler.py', 'sha256': 'b454acdd61de71c65bd4cff72853ffc44a2ef519d7b46a281d0f26f7c287887b'}, 'utils.py': {'url': 'https://raw.githubusercontent.com/nikhilsab/LLMFE/e9a58ca3416ae74dcc7549dffdf79b9289604091/utils.py', 'sha256': '64328cc283d77a73aea423352024bb9068e97dc904449ce71b566986cce3e1cc'}, 'preprocessing.py': {'url': 'https://raw.githubusercontent.com/nikhilsab/LLMFE/e9a58ca3416ae74dcc7549dffdf79b9289604091/preprocessing.py', 'sha256': 'd4e33caf2f264bc858328c4ba383816390920616bd3bbb63e13b6ccd7a5681da'}, 'prompts/domain_head.txt': {'url': 'https://raw.githubusercontent.com/nikhilsab/LLMFE/e9a58ca3416ae74dcc7549dffdf79b9289604091/prompts/domain_head.txt', 'sha256': '35afd562d53d48c14cda7d592533f9c6b8ae2b6d5a83232ae068839a8a7ceacf'}, 'prompts/domain_tail.txt': {'url': 'https://raw.githubusercontent.com/nikhilsab/LLMFE/e9a58ca3416ae74dcc7549dffdf79b9289604091/prompts/domain_tail.txt', 'sha256': '4bededf4edb06539c82fcf649b2cc6a77360a8a708c3eafa1ee77b41f65c20ce'}, 'prompts/operations_head.txt': {'url': 'https://raw.githubusercontent.com/nikhilsab/LLMFE/e9a58ca3416ae74dcc7549dffdf79b9289604091/prompts/operations_head.txt', 'sha256': '1cb75eba68eb103b068c061b93d3e8e634ca9ab8a9db50cfc40ef15656c5c72d'}, 'prompts/operations_tail.txt': {'url': 'https://raw.githubusercontent.com/nikhilsab/LLMFE/e9a58ca3416ae74dcc7549dffdf79b9289604091/prompts/operations_tail.txt', 'sha256': 'e6eaf3af08b3ba27e0bbb782bdb6bfeda4a26aa7e8a2fe8b64153051534274c3'}}}
import sys, importlib
CODE=WORK/'balance_diagnostic_runtime';CODE.mkdir(exist_ok=True)
for name,source in MODULES.items():(CODE/name).write_text(source,encoding='utf-8')
SOURCE=CODE/'upstream';SOURCE.mkdir(exist_ok=True)
for name,source in INPUT_FILES.items():
    path=SOURCE/name;path.parent.mkdir(parents=True,exist_ok=True);path.write_text(source,encoding='utf-8')
    assert hashlib.sha256(path.read_bytes()).hexdigest()==UPSTREAM_MANIFEST['files'][name]['sha256']
sys.path.insert(0,str(CODE))
import host as experiment
importlib.reload(experiment)
CONFIG['upstream_revision']=UPSTREAM_MANIFEST['revision']
CONFIG['upstream_manifest_sha256']=hashlib.sha256(json.dumps(UPSTREAM_MANIFEST,sort_keys=True).encode()).hexdigest()


In [ ]:
import torch
from huggingface_hub import HfApi, hf_hub_download
if not torch.cuda.is_available():
    raise RuntimeError('Включите GPU T4 в настройках Kaggle')
gpu = torch.cuda.get_device_properties(0)
if gpu.total_memory < 14 * 1024**3:
    raise RuntimeError('Нужно не менее14GiB GPU RAM; выбранная карта не подходит для этой конфигурации')

def load_kaggle_secret():
    from kaggle_secrets import UserSecretsClient
    return UserSecretsClient().get_secret('HF_TOKEN')

token = resolve_auth(MODEL_SOURCE, os.environ, load_kaggle_secret)
try:
    revision = HfApi(token=token).model_info(MODEL_ID, revision=MODEL_REVISION or 'main').sha
    hf_hub_download(MODEL_ID, 'config.json', revision=revision, token=token)
except Exception:
    hint = 'Internet и существование ревизии' if MODEL_SOURCE == 'unsloth' else 'Internet, HF_TOKEN и одобрение доступа Meta'
    raise RuntimeError('Не удалось прочитать ' + MODEL_ID + '. Проверьте ' + hint) from None

CONFIG['model_revision'] = revision
CONFIG['gpu'] = gpu.name
CONFIG['versions'] = {p:importlib.metadata.version(p) for p in
    ['torch','transformers','accelerate','bitsandbytes','xgboost-cpu','pandas','numpy','pyarrow']}
CONFIG['runtime_sha256'] = {name:hashlib.sha256(source.encode()).hexdigest() for name,source in MODULES.items()}
fingerprint = hashlib.sha256(json.dumps(CONFIG,sort_keys=True).encode()).hexdigest()[:12]
RUN_ID = CONFIG['protocol'] + '_' + fingerprint
OUT = WORK / RUN_ID
OUT.mkdir(exist_ok=True)
config_path = OUT / 'run_config.json'
if config_path.exists() and json.loads(config_path.read_text()) != CONFIG:
    raise RuntimeError('Config mismatch: start a new run directory')
experiment.write_json(config_path, CONFIG)
print('Run:', RUN_ID, '| source:', MODEL_SOURCE, '| model:', MODEL_ID, '| GPU:', gpu.name, '| revision:', revision)


In [ ]:
# Эти контроли ручные, не ответы Llama. Evaluator проверяется до загрузки больших весов.
experiment.write_json(OUT/'upstream_manifest.json',UPSTREAM_MANIFEST)
try:
    control_results=experiment.controls(SOURCE,OUT)
except Exception as error:
    archive=experiment.pack_results(OUT,error,stage='controls')
    display(FileLink(archive))
    print('Diagnostic failed BEFORE Llama generation. Failure ZIP:',archive,flush=True)
    raise


In [ ]:
try:
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
    set_seed(CONFIG['seed'])
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=revision, token=token, trust_remote_code=False)
    quantization = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=revision, token=token,
        quantization_config=quantization, torch_dtype=torch.float16, device_map={'':0},
        attn_implementation='sdpa', trust_remote_code=False, low_cpu_mem_usage=True)
    model.eval()
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = 'left'
    print('Model GPU footprint GiB:', model.get_memory_footprint()/1024**3)
    experiment.write_json(OUT / 'model_runtime.json', {
        'model':MODEL_ID, 'model_source':MODEL_SOURCE, 'revision':revision, 'is_loaded_in_4bit':bool(model.is_loaded_in_4bit),
        'gpu':gpu.name, 'footprint_bytes':int(model.get_memory_footprint()),
        'allocated_vram_bytes':int(torch.cuda.memory_allocated(0))})
except Exception as error:
    archive=experiment.pack_results(OUT,error,stage='model_load')
    display(FileLink(archive))
    print('Model load failed. Failure ZIP:',archive,flush=True)
    raise


In [ ]:
def generate_program(prompt, seed, mode='chat'):
    set_seed(seed)
    messages = [
        {'role':'system','content':'You are a feature engineering researcher. Follow the Python interface exactly.'},
        {'role':'user','content':prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True) if mode=='chat' else prompt
    inputs = tokenizer(text, return_tensors='pt', add_special_tokens=False).to('cuda:0')
    prompt_tokens = inputs['input_ids'].shape[1]
    if prompt_tokens > CONFIG['max_input_tokens']:
        raise ValueError('Prompt exceeds input budget; no silent context truncation')
    endings = [tokenizer.eos_token_id, tokenizer.convert_tokens_to_ids('<|eot_id|>')]
    endings = list({int(i) for i in endings if i is not None})
    tick = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=CONFIG['max_new_tokens'],
            do_sample=True, temperature=CONFIG['temperature'], top_p=CONFIG['top_p'], top_k=CONFIG['top_k'],
            num_return_sequences=1, eos_token_id=endings, pad_token_id=tokenizer.pad_token_id, use_cache=True)
    generated = output[0, prompt_tokens:]
    result = {'text':tokenizer.decode(generated, skip_special_tokens=True),
        'rendered_prompt':text, 'format':mode, 'seed':seed, 'prompt_tokens':int(prompt_tokens), 'generated_tokens':int(len(generated)),
        'seconds':time.perf_counter()-tick,
        'truncated':len(generated)>=CONFIG['max_new_tokens'] and int(generated[-1]) not in endings}
    del output, inputs, generated
    return result


In [ ]:
trials=experiment.run_diagnostic(SOURCE,OUT,CONFIG,generate_program)
del model
gc.collect();torch.cuda.empty_cache()
summary=experiment.report(OUT,CONFIG,trials,control_results)
display(pd.DataFrame(summary))
display(pd.read_csv(OUT/'trials.csv'))
display(Image(filename=str(OUT/'diagnostic.png')))
display(Markdown((OUT/'report.md').read_text()))
archive=experiment.pack_results(OUT)
display(FileLink(archive))
print('Download results ZIP: configuration, prompts, raw responses, errors and plots are included.')
